# Day 6: Session 6C - A Date Is Not a Number

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6c_dates.html)

Date: 09/08/2026

### explain why a date stored as text or as a number is not a date


In [2]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/toolik_weather.csv'
toolik = pd.read_csv(url)

toolik[['Year', 'Month', 'Date', 'Daily_AirTemp_Mean_C']].head()

,Year,Month,Date,Daily_AirTemp_Mean_C
0,1988,6,19880601,8.4
1,1988,6,19880602,6.0
2,1988,6,19880603,5.8
3,1988,6,19880604,1.8
4,1988,6,19880605,6.8


In [3]:
toolik['Date'].dtype

dtype('int64')

In [4]:
19880701 - 19880630

71

In [ ]:
# pd.to_datetime() takes a column of dates in disguise 
# and returns a column of real dates. 
# You describe the disguise with format=:

toolik['date'] = pd.to_datetime(toolik['Date'], format='%Y%m%d')

toolik[['Date', 'date']].head()

,Date,date
0,19880601,1988-06-01
1,19880602,1988-06-02
2,19880603,1988-06-03
3,19880604,1988-06-04
4,19880605,1988-06-05


### write the parsing pattern, pd.to_datetime(column, format=...), and build a format string out of %Y, %m and %d


In [1]:
# pd.to_datetime(column, format='%Y%m%d')
#              ↑               ↑
#              what to parse   how it is laid out

In [8]:
print(toolik['date'].dtype)
print(toolik['date'].min())
print(toolik['date'].max())

datetime64[ns]
1988-06-01 00:00:00
2018-12-31 00:00:00


### pull the year, month and day out of a parsed date with the .dt accessors


In [ ]:
# Every value in a parsed date column has a year, 
# a month and a day inside it, 
# and .dt is how you get at them:

toolik['year'] = toolik['date'].dt.year
toolik['month'] = toolik['date'].dt.month
toolik['day'] = toolik['date'].dt.day

toolik[['date', 'year', 'month', 'day']].head()

,date,year,month,day
0,1988-06-01,1988,6,1
1,1988-06-02,1988,6,2
2,1988-06-03,1988,6,3
3,1988-06-04,1988,6,4
4,1988-06-05,1988,6,5


In [10]:
print((toolik['year'] == toolik['Year']).all())
print((toolik['month'] == toolik['Month']).all())

True
True


### use a date component as a grouping key, and answer a question that needs one

In [11]:
# A date component is an ordinary column of integers, 
# so it is an ordinary grouping key, 
# and the split-apply-combine pattern works on it 

toolik.groupby('month')['Daily_AirTemp_Mean_C'].mean().round(2)

month
1    -22.89
2    -20.70
3    -20.69
4    -11.76
5     -0.80
6      8.59
7     11.22
8      7.23
9     -0.11
10   -10.54
11   -18.34
12   -21.44
Name: Daily_AirTemp_Mean_C, dtype: float64

In [12]:
toolik.groupby('year')['Daily_AirTemp_Mean_C'].agg(['count', 'mean']).head(3).round(2)

,count,mean
year,,
1988,214,-4.88
1989,365,-7.94
1990,365,-8.54


In [13]:
early = toolik[toolik['year'] <= 1998]
late = toolik[toolik['year'] >= 2009]

print(early.shape)
print(late.shape)

(3866, 25)
(3652, 25)


In [14]:
comparison = pd.DataFrame({
    'early': early.groupby('month')['Daily_AirTemp_Mean_C'].mean(),
    'late': late.groupby('month')['Daily_AirTemp_Mean_C'].mean(),
})

comparison['change'] = comparison['late'] - comparison['early']

comparison.round(2)


,early,late,change
month,,,
1,-24.72,-21.26,3.46
2,-21.86,-19.38,2.48
3,-18.62,-20.64,-2.02
4,-10.40,-11.86,-1.45
5,0.37,-0.54,-0.91
6,8.43,8.19,-0.24
7,12.05,11.28,-0.77
8,7.48,7.18,-0.30
9,-0.85,-0.01,0.84


### Key points
A date stored as text or as an integer is not a date. It will sort correctly and then fail at arithmetic, and it will not raise an error when it fails.

The parsing pattern is pd.to_datetime(column, format='...').

A format string is a picture of your data: %Y for a four-digit year, %m for a two-digit month, %d for a two-digit day, punctuation typed literally.

Always supply format=. It converts a silent wrong answer into a loud error.

.dt is to dates what .str is to text. .dt.year, .dt.month, .dt.day.

If .dt raises an AttributeError, your column has not been parsed yet.

Check your parse against anything you can: another column, a known date range, a row count.

Date components are ordinary columns, so they are ordinary grouping keys, which is usually the point of extracting them.